<a href="https://colab.research.google.com/github/yuhui-0611/ESAA/blob/main/ESAA_OB_WEEK01_1_Code_review.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 주제 및 데이터

< 난임 환자 대상 임신 성공 확률 예측 AI 오프라인 해커톤 >

[link text](https://dacon.io/competitions/official/236476/codeshare/12307?page=1&dtype=recent)



```
X_train, X_valid, y_train, y_valid = train_test_split(X_encoded, y_raw, test_size=0.2, random_state=42)

# ----------------------------
# PyTorch Dataset
# ----------------------------
class TabularDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_dataset = TabularDataset(X_train, y_train.values)
valid_dataset = TabularDataset(X_valid, y_valid.values)

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
valid_loader = DataLoader(valid_dataset, batch_size=256, shuffle=False)

# ----------------------------
# 모델 및 Loss 정의
# ----------------------------
class MLP(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.BatchNorm1d(128),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 1),
            nn.Sigmoid()
        )
    def forward(self, x):
        return self.net(x).squeeze()

class WeightedBrierLoss(nn.Module):
    def __init__(self, alpha=4.0):
        super().__init__()
        self.alpha = alpha

    def forward(self, preds, targets):
        weights = 1 + self.alpha * (targets-0.1) + (0.5 - targets).abs() ** 2
        return (weights * (preds - (targets-0.1)) ** 2).mean()

# ----------------------------
# 학습 루프 (상위 10개 모델 저장)
# ----------------------------
def train_model_topk(model, train_loader, valid_loader, loss_fn, optimizer, device, epochs=100, top_k=10):
    model.to(device)
    best_models = []

    for epoch in range(epochs):
        model.train()
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            preds = model(xb)
            loss = loss_fn(preds, yb)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        # 검증
        model.eval()
        val_preds = []
        val_targets = []
        with torch.no_grad():
            for xb, yb in valid_loader:
                xb = xb.to(device)
                preds = model(xb).cpu()
                val_preds.append(preds)
                val_targets.append(yb)
        val_preds = torch.cat(val_preds).numpy()
        val_targets = torch.cat(val_targets).numpy()
        score = competition_metric(val_targets, np.clip(val_preds, 0.05, 0.95))
        print(f"Epoch {epoch+1} | Competition Score: {score:.5f}")

        # top K 성능 기준 저장
        best_models.append((score, copy.deepcopy(model.state_dict())))
        best_models = sorted(best_models, key=lambda x: x[0], reverse=True)[:top_k]

    return best_models

# ----------------------------
# 학습 실행
# ----------------------------
input_dim = X_train.shape[1]
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = MLP(input_dim=input_dim)
loss_fn = WeightedBrierLoss(alpha=4.0)
#optimizer = torch.optim.Adam(model.parameters(), lr=3e-4)
optimizer = torch.optim.SGD(model.parameters(), lr=3e-3, momentum=0.9)


best_models = train_model_topk(model, train_loader, valid_loader, loss_fn, optimizer, device, epochs=100, top_k=10)

# ----------------------------
# 상위 10개 모델 앙상블 예측
# ----------------------------
test_tensor = torch.tensor(X_test_encoded, dtype=torch.float32).to(device)
ensemble_preds = np.zeros(len(test_tensor))

for i, (score, state_dict) in enumerate(best_models):
    print(f"✔️ 모델 {i+1} - Score: {score:.5f}")
    temp_model = MLP(input_dim)
    temp_model.load_state_dict(state_dict)
    temp_model.to(device)
    temp_model.eval()
    with torch.no_grad():
        preds = temp_model(test_tensor).cpu().numpy()
        ensemble_preds += preds / len(best_models)

# 확률 제한
test_preds = np.clip(ensemble_preds, 0.05, 0.95)
```



- Tabular 데이터로 MLP 모델을 학습하고,
성능이 좋은 상위 10개 모델을 골라서 → 그 10개 모델의 예측을 평균 내는 앙상블을 만든 뒤 → 예측 확률을 0.05~0.95 사이로 잘라서 제출용 예측을 만드는 코드

1.  train/valid 데이터 분리
2.  Dataset/DataLoader로 PyTorch용 데이터 구조 만들기
  - 한번에 전체 데이터를 넣지 않고, batch 단위로 나눠서 학습
3.  MLP 모델 정의 (Tabular용 2개 은닉층 + Sigmoid 출력)
  - 임신 성공 확률의 0 ~ 1 사이 확률 예측 문제라 nn.Sigmoid를 사용해 output을 0 ~ 1로 제한
4.  가중 Brier Loss 정의
5.  100 epoch 동안 학습하면서
  - 매 epoch마다 valid score 계산
  - 상위 10개 모델 파라미터 저장
6.  test 데이터에 대해
  - 상위 10개 모델 각각으로 예측
  - 예측 평균 = 앙상블
7 .  예측 확률을 0.05~0.95 범위로 잘라서 test_preds 완성 : 클리핑
  - 앙상블로 평균을 내서 예측을 부드럽게 만든 뒤, 추가로 클리핑으로 극단 확률을 잘라 대회 평가 지표에 더 유리한 확률 분포로 만듦
  - Brier Score / Logloss 같은 지표는 극단적 확률에 민감하기 때문

# 새롭게 알게 된 내용

< Brier Score >
- = 모델이 예측한 “확률값”이 정답과 얼마나 가까운지를 측정하는 지표
- 예측이 확률일 때만 사용 가능
- 해석 : 0에 가까울수록 좋음
- 일반 accuracy나 F1처럼 “맞았냐 틀렸냐”가 아니라 예측 확률 그 자체가 얼마나 정답에 가까운지를 평가

< 시그모이드 >
- = 입력값(−∞~+∞)을 0~1 사이의 값으로 압축해주는 수학 함수
- 즉, '아무리 큰 숫자가 와도 → 1에 가까운 값, 아무리 작은 숫자가 와도 → 0에 가까운 값'으로 만들어주기에 확률 표현에 good
- 이진 확률 문제에 사용

< 클리핑 >
- 시그모이드의 단점 보완
> 끝부분(0 또는 1 근처)에서 변화가 거의 없음